In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import pickle
import os
from torch_geometric.datasets import BA2MotifDataset
from torch.utils.data import Subset
from torch_geometric.data import Data
import networkx as ntx
from torch_geometric.utils import to_networkx,subgraph
import matplotlib.pyplot as plt

In [44]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [45]:
dataset_path='./data/BA2Motif'
with open('./data/splits.pkl', 'rb') as f:
    splits=pickle.load(f)

full_dataset=BA2MotifDataset(root=dataset_path)
test_dataset=Subset(full_dataset,splits['test'])

with open('./data/test_result.pkl', 'rb') as f:
    test_result=pickle.load(f)


In [46]:
print(test_result['correct'][0])

{'index': 0, 'pred': 0}


In [47]:
class GINGraphClf(nn.Module):
    def __init__(self, in_dim,out_dim, hidden_dim=64):
        super().__init__()
        self.conv1=GINConv(nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        self.conv2=GINConv(nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        self.conv3=GINConv(nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        
        self.mlp = nn.Sequential(
            nn.Linear(3*hidden_dim, hidden_dim//2),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, out_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x, edge_index, batch):
        h1=self.conv1(x, edge_index)
        h2=self.conv2(h1, edge_index)
        h3=self.conv3(h2, edge_index)

        h1=global_add_pool(h1,batch)
        h2=global_add_pool(h2,batch)
        h3=global_add_pool(h3,batch)    


        x=torch.cat([h1,h2,h3],dim=1)


        return self.mlp(x)

In [48]:
model=GINGraphClf(test_dataset[0].num_features,2).to(device)


checkpoint=torch.load('./models/GIN/best_gin_model.pt',map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [ ]:
def remove_node(data, nodes_to_remove):

    if isinstance(nodes_to_remove, list):
        nodes_to_remove = torch.tensor(nodes_to_remove, dtype=torch.long)
    num_nodes = data.num_nodes
    keep_mask = torch.ones(num_nodes, dtype=torch.bool)
    keep_mask[nodes_to_remove] = False
    new_x = data.x[keep_mask]
    
    new_index = torch.zeros(num_nodes, dtype=torch.long)
    new_index[keep_mask] = torch.arange(keep_mask.sum().item())
    
    edge_index = data.edge_index
    edge_mask = keep_mask[edge_index[0]] & keep_mask[edge_index[1]]
    new_edge_index = edge_index[:, edge_mask]
    new_edge_index = new_index[new_edge_index]
    
    new_data = Data(x=new_x, edge_index=new_edge_index)
    
    for key in data.keys():   
        if key in ['x', 'edge_index', 'num_nodes']:
            continue
        attr = data[key]
        if torch.is_tensor(attr) and attr.size(0) == num_nodes:
            new_data[key] = attr[keep_mask]
        else:
            new_data[key] = attr
    return new_data

In [ ]:
def subgraph_connection_check(graph,node_set):
    if len(node_set) <= 1:
        return True
    sub_edge_index, _ = subgraph(node_set, graph.edge_index, relabel_nodes=True)
    sub_data = Data(edge_index=sub_edge_index, num_nodes=len(node_set))
    G = to_networkx(sub_data, to_undirected=True)
    return ntx.is_connected(G)

In [51]:
def compute_score(model,data,target_class,device):
    model.eval()
    with torch.no_grad():
        if data.batch is not None:
            out=model(data.x,data.edge_index,data.batch)
        else:
            batch=torch.zeros(data.num_nodes,dtype=torch.long,device=device)
            out = model(data.x, data.edge_index, batch)
        if out.dim() == 1:
            out = out.unsqueeze(0)
        score = out[0, target_class]  
    return score 



In [ ]:
def subgraph_explanation(model, data, target_class, budget=0.2, device=device, score_func='logit'):
    
    
    current_nodes = set(range(data.num_nodes))
    removed_nodes = []
    
    initial_num_nodes = data.num_nodes
    target_size = budget if isinstance(budget, int) else int(initial_num_nodes * budget)
    target_size = max(1, min(target_size, initial_num_nodes))

    while len(current_nodes) > target_size:
        best_node = None
        best_delta = None
        best_new_data = None
        
        for v in list(current_nodes):
            candidate_nodes = current_nodes - {v}
            if not subgraph_connection_check(data, list(candidate_nodes)):
                continue 
            
            current_subgraph = remove_node(data, list(set(range(data.num_nodes)) - current_nodes))
            candidate_subgraph = remove_node(data, list(set(range(data.num_nodes)) - candidate_nodes))
            
            score_current = compute_score(model, current_subgraph, target_class, device)
            score_candidate = compute_score(model, candidate_subgraph, target_class, device)
            
            delta = score_current - score_candidate
            
            if best_delta is None or delta < best_delta:
                best_delta = delta
                best_node = v
                best_new_data = candidate_subgraph
        
        if best_node is None:
            print("there is no node to remove! perhaps graph is not connected at all")
            break
        
        current_nodes.remove(best_node)
        removed_nodes.append(best_node)

    final_subgraph = remove_node(data, list(set(range(data.num_nodes)) - current_nodes))
    return final_subgraph, removed_nodes

In [ ]:
def visualize_explanation(original_data, explanation_node_indices, title="Explanation Subgraph", save_path=None):

    G = to_networkx(original_data, to_undirected=True)
    pos = ntx.spring_layout(G, seed=404131029)  
    
    node_colors = []
    for node in G.nodes():
        if node in explanation_node_indices:
            node_colors.append('red')
        else:
            node_colors.append('skyblue')
    
    plt.figure(figsize=(10, 8))
    ntx.draw(G, pos, node_color=node_colors, with_labels=True, 
            node_size=500, font_size=10, font_weight='bold', 
            edge_color='gray', alpha=0.8)
    
    plt.title(f"{title}\n(Explanation nodes: {len(explanation_node_indices)} / {original_data.num_nodes})")
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    


In [54]:
print(torch.randint(0,len(test_result['correct'])-1,(1,)).item()
)

85


In [55]:
random_index=torch.randint(0,len(test_result['correct'])-1,(1,)).item()
data=test_dataset[test_result['correct'][random_index]['index']]
pred_class=test_result['correct'][random_index]['pred']


In [56]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [57]:
for key in test_result.keys():
    print(key)
    print(len(test_result[key]))

correct
100
wrong
0


In [68]:
for key in test_result.keys():
    if len(test_result[key]) == 0:
        continue
    max_pic=10
    if len(test_result[key])<max_pic:
        max_pic=len(test_result[key])
    for i in range(max_pic):
        random_index=torch.randint(0,len(test_result['correct'])-1,(1,)).item()

        data=test_dataset[test_result[key][random_index]['index']]
        pred_class=test_result[key][random_index]['pred']
        explanation_graph, removed_nodes = subgraph_explanation(
            model, data, target_class=pred_class, budget=0.2, device=device
        )
        all_nodes = set(range(data.num_nodes))
        remaining_nodes = list(all_nodes - set(removed_nodes))
        visualize_explanation(data.cpu(), remaining_nodes,save_path=f"./explanation/{key}/index_{test_result[key][random_index]['index']}.png")  # حتماً داده را به CPU ببرید
